# Comprehensive Language Embedding Analysis

This notebook consolidates the entire pipeline for analyzing Llama-3 language embeddings, from data preprocessing and embedding generation to advanced clustering and evaluation.

## Objectives:
1.  **Data Preprocessing**: Filter FineWeb-2 languages and assign resource tiers.
2.  **Embedding Generation**: Generate semantic embeddings using Llama-3.2-1B.
3.  **Dimensionality Reduction**: Visualize embeddings using UMAP and t-SNE.
4.  **Clustering**: Apply K-Means, HDBSCAN, and Hierarchical Clustering to identify language families.
5.  **Evaluation**: Assess clustering quality against genealogical data using rigorous metrics (AMI, NMI, ARI) and interactive visualizations.

In [72]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import umap
import hdbscan
import plotly.graph_objects as go
from enum import Enum
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples, pairwise_distances
from sklearn.metrics import homogeneity_score, adjusted_mutual_info_score, normalized_mutual_info_score, adjusted_rand_score, confusion_matrix
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

PROJECT_ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
if PROJECT_ROOT_DIR not in sys.path:
    sys.path.append(PROJECT_ROOT_DIR)

DATA_DIR = os.path.join(os.getcwd(), "processed_artifacts")
BASE_DATA_DIR = os.path.join(os.getcwd(), "base_data")
os.makedirs(DATA_DIR, exist_ok=True)

from approaches.CoLA.distributed_data_processor.language_subsets import fineweb2_benchmark_languages

print("Environment Setup Complete.")

Environment Setup Complete.


## Part 1: Data Preprocessing
Load FineWeb-2 metadata, filter for benchmark languages, and attach resource categories.

In [73]:
class ResourceCategory(str, Enum):
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"

def load_and_filter_metadata(csv_path, benchmark_languages):
    df = pd.read_csv(csv_path)
    df = df[
        (~df["subset"].astype(str).str.endswith("_removed"))
        & (df["split"] == "train")
        & (df["family"] != "-")
        & (df["subset"].isin(benchmark_languages))
    ].reset_index(drop=True)
    print(f"Loaded {len(df)} evaluable languages.")
    return df

def add_resource_categories(df, resource_csv_path):
    res_df = pd.read_csv(resource_csv_path, sep="\t")
    res_df = res_df[~res_df["resource_category"].str.endswith("*", na=False)]
    df["_lc"] = df["subset"].str.lower()
    res_df["_lc"] = res_df["lang_code"].str.lower()

    df = df.merge(res_df[["_lc", "resource_category"]], on="_lc", how="left") \
           .drop(columns="_lc") \
           .sort_values("documents", ascending=False) \
           .reset_index(drop=True)

    last = df.groupby("resource_category", dropna=False).tail(1)
    idx = {r["resource_category"]: r.name for _, r in last.iterrows()}

    MAP = {
        "high": ResourceCategory.HIGH,
        "medhigh": ResourceCategory.MEDIUM,
        "medlow": ResourceCategory.LOW,
        "low": ResourceCategory.LOW,
        "not_enough": ResourceCategory.LOW,
    }

    bounds = sorted(
        [(MAP.get(k, ResourceCategory.LOW).value, v) for k, v in idx.items()],
        key=lambda x: x[1]
    )

    start = 0
    for cat, end in bounds:
        df.loc[start:end, "resource_category"] = cat
        start = end + 1
    if start < len(df):
        df.loc[start:, "resource_category"] = df["resource_category"].ffill()
    return df

metadata = load_and_filter_metadata(
    os.path.join(BASE_DATA_DIR, "fineweb2-language-distribution.csv"),
    fineweb2_benchmark_languages
)
metadata = add_resource_categories(
    metadata,
    os.path.join(BASE_DATA_DIR, "lang_resource_dataset.tsv")
)

print("\nResource tiers:")
print(metadata["resource_category"].value_counts())
print("\nTop families:")
print(metadata["family"].value_counts().head(10))


Loaded 191 evaluable languages.

Resource tiers:
resource_category
low       135
high       31
medium     25
Name: count, dtype: int64

Top families:
family
Indo-European    76
Niger-Congo      32
Austronesian     18
Afro-Asiatic     18
Turkic           11
Sino-Tibetan      6
Creole            5
Dravidian         4
Nilo-Saharan      4
Kra-Dai           3
Name: count, dtype: int64


## Part 2: Embedding Generation (Llama-3)
Generate embeddings using `meta-llama/Llama-3.2-1B`. If embeddings already exist in `processed_artifacts/llm_embeddings.csv`, they will be loaded to save time.

In [74]:
flores_df = pd.read_csv(os.path.join(DATA_DIR, "flores_embeddings.csv"))
embedding_cols = sorted(
    [c for c in flores_df.columns if c.startswith("llm_emb_")],
    key=lambda x: int(x.split("_")[-1])
)

flores_embeddings = flores_df[["subset"] + embedding_cols].drop_duplicates("subset")

missing = sorted(set(metadata["subset"]) - set(flores_embeddings["subset"]))
if missing:
    print(f"Warning: {len(missing)} metadata subsets missing embeddings ({', '.join(missing[:5])}...).")

metadata = (
    metadata.set_index("subset")
    .join(flores_embeddings.set_index("subset"), how="inner")
    .reset_index()
)

X = metadata[embedding_cols].to_numpy()
print("Embedding Matrix Shape:", X.shape)


Embedding Matrix Shape: (184, 4096)


/tmp/ipykernel_930244/3693342172.py:1: DtypeWarning:

Columns (12,13) have mixed types. Specify dtype option on import or set low_memory=False.



In [76]:
from sklearn.metrics.pairwise import cosine_distances

print("Running HDBSCAN with precomputed cosine distances...")

# 1. Compute cosine distance matrix (1 - cosine similarity)
D = cosine_distances(X)   # shape: (n_samples, n_samples)

# 2. Run HDBSCAN with precomputed distances
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="precomputed",
    gen_min_span_tree=True
)

metadata["hdbscan_cluster"] = clusterer.fit_predict(D)

# 3. Count clusters (excluding noise label -1)
n_clusters = len(set(metadata["hdbscan_cluster"])) - (1 if -1 in metadata["hdbscan_cluster"] else 0)
print(f"HDBSCAN found {n_clusters} clusters.")


Running HDBSCAN with precomputed cosine distances...
HDBSCAN found 6 clusters.


### Clustering Pipeline Overview
1. Filter FineWeb-2 metadata down to benchmark subsets and label each language with a resource tier.
2. Attach the FLORES mean-pooled embeddings from `processed_artifacts/flores_embeddings.csv` so every surviving row has a 4,096-d vector.
3. Compute cosine distances across the embedding matrix and feed them into HDBSCAN (`min_cluster_size=10`, `min_samples=5`), which yields the coarse language clusters.
4. Project the embeddings with t-SNE and UMAP so we can inspect the clusters (colored either by family or by the discovered cluster ids).

In [77]:
print("Computing t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, metric="cosine")
X_tsne = tsne.fit_transform(X)
metadata["TSNE1"] = X_tsne[:, 0]
metadata["TSNE2"] = X_tsne[:, 1]

print("Computing UMAP...")
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, metric="cosine")
X_umap = reducer.fit_transform(X)
metadata["UMAP1"] = X_umap[:, 0]
metadata["UMAP2"] = X_umap[:, 1]


Computing t-SNE...
Computing UMAP...


/home/ubuntu/miniconda3/envs/cola_llama_factory/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [78]:
def plot_projection(projection_name, x_col, y_col, color_by="family"):
    fig = go.Figure()

    # Choose coloring column
    color_col = metadata[color_by].astype('category').cat.codes

    fig.add_trace(go.Scatter(
        x=metadata[x_col],
        y=metadata[y_col],
        mode='markers',
        marker=dict(
            size=8,
            color=color_col,
            colorscale='Turbo',
            opacity=0.7,
            showscale=True
        ),
        text=metadata["name"],
        customdata=metadata[[x_col, y_col, "family", "resource_category"]],
        hovertemplate=create_hover_template(projection_name),
        name='Languages'
    ))
    fig.update_layout(
        title=f"{projection_name} Projection (colored by {color_by})",
        width=1000,
        height=700,
        hoverlabel=dict(bgcolor="white", font_size=12)
    )
    fig.show()


In [79]:
plot_projection("t-SNE", "TSNE1", "TSNE2", color_by="family")
plot_projection("UMAP", "UMAP1", "UMAP2", color_by="family")

plot_projection("t-SNE", "TSNE1", "TSNE2", color_by="hdbscan_cluster")
plot_projection("UMAP", "UMAP1", "UMAP2", color_by="hdbscan_cluster")
